# Fadnavis Media Monitor — sitemap-driven

Replaces the Selenium/Google-SERP scraper. Articles are discovered from each
publisher's **news sitemap** (the feed they maintain for crawlers), filtered to
a date window, then keyword-matched.

**How it behaves**
- `robots.txt` is checked for every URL before fetching (via `protego`); disallowed URLs are skipped.
- Requests are rate limited per domain, with exponential backoff on 429/5xx.
- Pages are cached under `.cache/pages`, so re-runs are nearly free.

**Coverage caveat** — most Indian publishers expose only the last ~48h in their
news sitemaps. ABP Live shards by date and backfills ~30 days; the rest are
current-window only. So: run this **daily** to build history going forward.

In [ ]:
!pip install -q requests beautifulsoup4 lxml protego pandas openpyxl

In [ ]:
import pandas as pd
import news_monitor as nm

pd.set_option("display.max_colwidth", 60)

## Config

In [ ]:
START_DATE = "08/30/2026"
END_DATE   = "08/31/2026"

# Identify yourself honestly; some sites rate-limit unknown agents.
USER_AGENT = "MediaMonitor/1.0 (+contact: aiml@strelema.com)"

DELAY      = 2.0    # seconds between requests to the same domain
DEEP       = True   # also match keywords in article body text
DEEP_LIMIT = 400    # max article pages fetched per run

OUTPUT = f"DF_{START_DATE[:2]}{START_DATE[3:5]}_{END_DATE[:2]}{END_DATE[3:5]}_News.xlsx"

## Search terms

Terms come from `profiles.json` — one profile per entity, each with its search
terms and report brief. Pick a profile by id (`devendra-fadnavis`,
`ravindra-chavan`, ...) or set `terms` by hand.

Keep terms short. Full-name and distinctive-surname forms are what actually
match a headline or URL slug; long phrases like `"Devendra Fadnavis latest
news"` were Google *query* strings and almost never appear verbatim in an
article.

In [ ]:
import profiles

PROFILE = "devendra-fadnavis"

p = profiles.get(PROFILE)
terms   = p["terms"]
SUBJECT = p["subject"]

print(f"{p['name']}: {len(terms)} terms")
print(terms)
print("\nAvailable:", ", ".join(x["id"] for x in profiles.load()))

## Sources

In [ ]:
for s in nm.SOURCES:
    kind = "dated shards" if s.dated_sitemap else "current window only"
    print(f"{s.name:<20} {s.language:<6} {kind}")

## Run

In [ ]:
df = nm.run(
    keywords   = terms,
    start_date = START_DATE,
    end_date   = END_DATE,
    user_agent = USER_AGENT,
    deep       = DEEP,
    deep_limit = DEEP_LIMIT,
    delay      = DELAY,
)
df.head(20)

## Save

In [ ]:
df.to_excel(OUTPUT, index=False)
print(f"{len(df)} articles -> {OUTPUT}")

if not df.empty:
    print("\nBy source:");     print(df["Source"].value_counts().to_string())
    print("\nMatched where:"); print(df["Matched In"].value_counts().to_string())

## Daily run

Because sitemaps only reach back ~48h, schedule this to run daily and append.
The URL de-dupe makes re-runs safe.

In [ ]:
from pathlib import Path
from datetime import date

MASTER = Path("fadnavis_master.xlsx")

def append_to_master(new_df, path=MASTER):
    if path.exists():
        combined = pd.concat([pd.read_excel(path), new_df], ignore_index=True)
    else:
        combined = new_df
    combined.drop_duplicates(subset=["Links"], inplace=True)
    combined.sort_values("Published", ascending=False, inplace=True)
    combined.to_excel(path, index=False)
    return combined

# master = append_to_master(df)
# print(f"master now holds {len(master)} articles")